In [1]:
!pip install biotite
!pip install rdkit
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.8 MB/s eta 0:00:00


In [2]:
# ============================================================
# Stanford RNA 3D Folding — TBM with External Templates
# Based on 0.371 LB notebook + External Template Library
# ============================================================

!pip install --no-index /kaggle/input/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl

import pandas as pd
import numpy as np
import random
import time
import warnings
import os
import sys
import json
import gzip
import glob
import gc

# Determinism hygiene
os.environ["PYTHONHASHSEED"] = "0"
warnings.filterwarnings("ignore")

seed = 42
np.random.seed(seed)
random.seed(seed)

# -----------------------------
# 1) Data ingestion
# -----------------------------
DATA_PATH = "/kaggle/input/stanford-rna-3d-folding-2/"
train_seqs = pd.read_csv("/kaggle/input/competitions/stanford-rna-3d-folding-2/train_sequences.csv")
test_seqs = pd.read_csv("/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv")
train_labels = pd.read_csv("/kaggle/input/competitions/stanford-rna-3d-folding-2/train_labels.csv")

# ★★ 新增: 加载validation数据
validation_seqs = pd.read_csv("/kaggle/input/competitions/stanford-rna-3d-folding-2/validation_sequences.csv")
validation_labels = pd.read_csv("/kaggle/input/competitions/stanford-rna-3d-folding-2/validation_labels.csv")

sys.path.append(os.path.join(DATA_PATH, "extra"))

try:
    import typing as _typing
    import builtins as _builtins

    _builtins.Dict = getattr(_typing, "Dict")
    _builtins.Tuple = getattr(_typing, "Tuple")
    _builtins.List = getattr(_typing, "List")

    from parse_fasta_py import parse_fasta as _parse_fasta_raw

    def parse_fasta(fasta_content: str):
        d = _parse_fasta_raw(fasta_content)
        out = {}
        for k, v in d.items():
            out[k] = v[0] if isinstance(v, tuple) else v
        return out

except Exception:
    def parse_fasta(fasta_content: str):
        out = {}
        cur = None
        seq_parts = []
        for line in str(fasta_content).splitlines():
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur is not None:
                    out[cur] = "".join(seq_parts)
                header = line[1:]
                cur = header.split()[0]
                seq_parts = []
            else:
                seq_parts.append(line.replace(" ", ""))
        if cur is not None:
            out[cur] = "".join(seq_parts)
        return out

def parse_stoichiometry(stoich: str):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    out = []
    for part in str(stoich).split(";"):
        ch, cnt = part.split(":")
        out.append((ch.strip(), int(cnt)))
    return out

def get_chain_segments(row):
    seq = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_seq = row.get("all_sequences", "")

    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip() == "" or str(all_seq).strip() == "":
        return [(0, len(seq))]

    try:
        chain_dict = parse_fasta(all_seq)
        order = parse_stoichiometry(stoich)

        segs = []
        pos = 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                L = len(base)
                segs.append((pos, pos + L))
                pos += L

        if pos != len(seq):
            return [(0, len(seq))]
        return segs
    except Exception:
        return [(0, len(seq))]

def build_segments_map(df):
    seg_map = {}
    stoich_map = {}
    for _, r in df.iterrows():
        tid = r["target_id"]
        seg_map[tid] = get_chain_segments(r)
        stoich_map[tid] = str(r.get("stoichiometry", "") if not pd.isna(r.get("stoichiometry", "")) else "")
    return seg_map, stoich_map

train_segs_map, train_stoich_map = build_segments_map(train_seqs)
test_segs_map, test_stoich_map = build_segments_map(test_seqs)

# -----------------------------
# 2) Labels to templates
# -----------------------------
def process_labels(labels_df: pd.DataFrame):
    coords_dict = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for id_prefix, group in labels_df.groupby(prefixes, sort=False):
        coords_dict[id_prefix] = group.sort_values("resid")[["x_1", "y_1", "z_1"]].values
    return coords_dict

train_coords_dict = process_labels(train_labels)
valid_coords_dict = process_labels(validation_labels)

# ★★ 合并train和validation
all_train_seqs = pd.concat([train_seqs, validation_seqs], ignore_index=True)
all_train_coords = {**train_coords_dict, **valid_coords_dict}

print(f"✓ Template pool (train+val): {len(all_train_coords)} structures")

# -----------------------------
# 3) ★★ External Template Library (新增)
# -----------------------------
def load_json_coords_fixed(json_file):
    """
    Load RNA sequence and C1' coordinates from JSON template file.
    """
    try:
        if json_file.endswith(".gz"):
            with gzip.open(json_file, "rt") as f:
                data = json.load(f)
        else:
            with open(json_file) as f:
                data = json.load(f)

        key = list(data.keys())[0]
        residues = data[key]

        if not isinstance(residues, list) or len(residues) == 0:
            return None, None

        seq = []
        coords = []

        for r in residues:
            if not isinstance(r, dict):
                continue
            
            one_letter = r.get("one_letter_code", "")
            if not one_letter:
                continue
            seq.append(one_letter)

            atoms = r.get("atoms", {})
            if "C1'" in atoms:
                c1_coord = atoms["C1'"]
                if isinstance(c1_coord, list) and len(c1_coord) == 3:
                    coords.append([float(c1_coord[0]), float(c1_coord[1]), float(c1_coord[2])])
                else:
                    coords.append([np.nan, np.nan, np.nan])
            else:
                coords.append([np.nan, np.nan, np.nan])

        if len(seq) == 0:
            return None, None

        seq_str = "".join(seq).upper().replace("T", "U")
        coords_arr = np.array(coords, dtype=np.float64)

        valid_mask = ~np.isnan(coords_arr[:, 0])
        if valid_mask.sum() < max(3, 0.3 * len(seq_str)):
            return None, None

        rna_frac = sum(1 for c in seq_str if c in 'ACGU') / len(seq_str)
        if rna_frac < 0.7:
            return None, None

        return seq_str, coords_arr

    except Exception:
        return None, None

# ★★ External template paths
TBM_ROOT_ALTERNATIVES = [
    "/kaggle/input/datasets/odat1248/rnatbm-templates-20250521-re/fortbm_clustered_sampled_20250526_gt_re",
    "/kaggle/input/datasets/odat1248/rnatbm2025-set/rna_tbm2025_clustered/fortbm_clustered_sampled_20250526_gt_0521",
    # 可根据实际路径调整
]

ext_loaded = 0
existing_seqs = set(all_train_seqs['sequence'].values)

valid_tbm_roots = []
for tbm_path in TBM_ROOT_ALTERNATIVES:
    if os.path.exists(tbm_path):
        valid_tbm_roots.append(tbm_path)
        print(f"★★ Found external template root: {tbm_path}")

if valid_tbm_roots:
    json_files = []
    for tbm_root in valid_tbm_roots:
        files = glob.glob(tbm_root + "/**/*.json*", recursive=True)
        json_files.extend(files)
        print(f"  {tbm_root}: {len(files)} JSON files")
    
    print(f"  Total: {len(json_files)} JSON files")
    
    new_rows = []
    failed = 0
    parsed_ok = 0
    
    for f in json_files:
        seq, coords = load_json_coords_fixed(f)
        
        if seq is None:
            failed += 1
            continue
        
        parsed_ok += 1
        
        if seq in existing_seqs:
            continue
        
        ext_id = f"EXT_{abs(hash(seq)) % 10**8:08d}"
        
        all_train_coords[ext_id] = coords
        new_rows.append({'target_id': ext_id, 'sequence': seq})
        existing_seqs.add(seq)
    
    print(f"  Parsed OK: {parsed_ok} | Failed: {failed} | New unique: {len(new_rows)}")
    
    if new_rows:
        ext_df = pd.DataFrame(new_rows)
        all_train_seqs = pd.concat([all_train_seqs, ext_df], ignore_index=True)
        ext_loaded = len(new_rows)
        print(f"  ✓ Added {len(new_rows)} unique external templates")

else:
    print("\n★★ External template roots NOT found - using train+val only")

print(f"\n★★ Final template pool: {len(all_train_coords)} structures")

del existing_seqs
gc.collect()

# -----------------------------
# 4) Similarity search
# -----------------------------
from Bio.Align import PairwiseAligner

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1.5
aligner.open_gap_score = -8
aligner.extend_gap_score = -0.4
aligner.query_left_open_gap_score = -8
aligner.query_left_extend_gap_score = -0.4
aligner.query_right_open_gap_score = -8
aligner.query_right_extend_gap_score = -0.4
aligner.target_left_open_gap_score = -8
aligner.target_left_extend_gap_score = -0.4
aligner.target_right_open_gap_score = -8
aligner.target_right_extend_gap_score = -0.4

def find_similar_sequences(query_seq: str, train_seqs_df: pd.DataFrame, train_coords_dict: dict, top_n: int = 5):
    similar_seqs = []
    for _, row in train_seqs_df.iterrows():
        target_id = row["target_id"]
        train_seq = row["sequence"]
        if target_id not in train_coords_dict:
            continue

        if abs(len(train_seq) - len(query_seq)) / max(len(train_seq), len(query_seq)) > 0.3:
            continue

        raw_score = aligner.score(query_seq, train_seq)
        normalized_score = raw_score / (2 * min(len(query_seq), len(train_seq)))
        similar_seqs.append((target_id, train_seq, normalized_score, train_coords_dict[target_id]))

    similar_seqs.sort(key=lambda x: x[2], reverse=True)
    return similar_seqs[:top_n]

# -----------------------------
# 5) Template transfer
# -----------------------------
def adapt_template_to_query(query_seq: str, template_seq: str, template_coords: np.ndarray):
    alignment = next(iter(aligner.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan, dtype=float)

    for (q_start, q_end), (t_start, t_end) in zip(*alignment.aligned):
        t_chunk = template_coords[t_start:t_end]
        if len(t_chunk) == (q_end - q_start):
            new_coords[q_start:q_end] = t_chunk

    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            prev_v = next((j for j in range(i - 1, -1, -1) if not np.isnan(new_coords[j, 0])), -1)
            next_v = next((j for j in range(i + 1, len(new_coords)) if not np.isnan(new_coords[j, 0])), -1)

            if prev_v >= 0 and next_v >= 0:
                w = (i - prev_v) / (next_v - prev_v)
                new_coords[i] = (1 - w) * new_coords[prev_v] + w * new_coords[next_v]
            elif prev_v >= 0:
                new_coords[i] = new_coords[prev_v] + [3, 0, 0]
            elif next_v >= 0:
                new_coords[i] = new_coords[next_v] + [3, 0, 0]
            else:
                new_coords[i] = [i * 3, 0, 0]

    return np.nan_to_num(new_coords)

# -----------------------------
# 6) ★★ Kabsch alignment & Ensemble centroid (新增)
# -----------------------------
def kabsch(P, Q):
    """Kabsch algorithm for optimal rotation"""
    assert P.shape == Q.shape
    mask = np.isfinite(P[:, 0]) & np.isfinite(Q[:, 0])
    Pm, Qm = P[mask], Q[mask]
    if len(Pm) < 3:
        return None, None, None
    Pc, Qc = Pm.mean(0), Qm.mean(0)
    H = (Pm - Pc).T @ (Qm - Qc)
    V, S, W = np.linalg.svd(H)
    d = np.sign(np.linalg.det(V @ W))
    U = V @ np.diag([1, 1, d]) @ W
    return U, Pc, Qc

def ensemble_centroid(preds):
    """Compute median centroid after Kabsch alignment"""
    if not preds or len(preds) == 0:
        return None
    if len(preds) == 1:
        return preds[0].copy()

    ref = preds[0]
    aligned = [ref.copy()]

    for p in preds[1:]:
        result = kabsch(p.copy(), ref.copy())
        if result[0] is not None:
            U, Pc, Qc = result
            aligned.append((p - Pc) @ U + Qc)
        else:
            aligned.append(p.copy())

    return np.median(np.stack(aligned), axis=0)

def hybrid_confidence(coords: np.ndarray) -> float:
    """Geometric confidence scorer"""
    if coords is None or len(coords) < 3:
        return -1e9

    valid = np.sum(np.isfinite(coords[:, 0]))
    if valid < 3:
        return -1e9

    coverage = valid / len(coords)

    finite_mask = np.isfinite(coords[:, 0])
    finite_coords = coords[finite_mask]
    diffs = np.linalg.norm(np.diff(finite_coords, axis=0), axis=1)

    if len(diffs) == 0:
        return -1e9

    smooth = np.std(diffs)
    mean_step = np.mean(diffs)
    spacing_error = abs(mean_step - 5.95)

    center = finite_coords.mean(axis=0)
    compact = np.mean(np.linalg.norm(finite_coords - center, axis=1))

    return coverage * 2.0 - smooth * 0.3 - spacing_error * 0.2 - 0.02 * compact

# -----------------------------
# 7) Segment-aware local refinement
# -----------------------------
def adaptive_rna_constraints(coordinates: np.ndarray, target_id: str, confidence: float = 1.0, passes: int = 2):
    coords = coordinates.copy()
    segments = test_segs_map.get(target_id, [(0, len(coords))])

    strength = 0.75 * (1.0 - min(confidence, 0.90))
    strength = max(strength, 0.02)

    for _ in range(passes):
        for (s, e) in segments:
            X = coords[s:e]
            L = e - s
            if L < 3:
                coords[s:e] = X
                continue

            d = X[1:] - X[:-1]
            dist = np.linalg.norm(d, axis=1) + 1e-6
            target = 5.95
            scale = (target - dist) / dist
            adj = (d * scale[:, None]) * (0.22 * strength)
            X[:-1] -= adj
            X[1:] += adj

            d2 = X[2:] - X[:-2]
            dist2 = np.linalg.norm(d2, axis=1) + 1e-6
            target2 = 10.2
            scale2 = (target2 - dist2) / dist2
            adj2 = (d2 * scale2[:, None]) * (0.10 * strength)
            X[:-2] -= adj2
            X[2:] += adj2

            lap = 0.5 * (X[:-2] + X[2:]) - X[1:-1]
            X[1:-1] += (0.06 * strength) * lap

            if L >= 25:
                k = min(L, 160) if L > 220 else L
                idx = np.linspace(0, L - 1, k).astype(int) if k < L else np.arange(L)

                P = X[idx]
                diff = P[:, None, :] - P[None, :, :]
                distm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])

                mask = (sep > 2) & (distm < 3.2)
                if np.any(mask):
                    force = (3.2 - distm) / distm
                    vec = (diff * force[:, :, None] * mask[:, :, None]).sum(axis=1)
                    X[idx] += (0.015 * strength) * vec

            coords[s:e] = X

    return coords

# -----------------------------
# 8) Geometric transforms for diversity
# -----------------------------
def _rotmat(axis, ang):
    axis = np.asarray(axis, float)
    axis = axis / (np.linalg.norm(axis) + 1e-12)
    x, y, z = axis
    c, s = np.cos(ang), np.sin(ang)
    C = 1.0 - c
    return np.array(
        [
            [c + x * x * C, x * y * C - z * s, x * z * C + y * s],
            [y * x * C + z * s, c + y * y * C, y * z * C - x * s],
            [z * x * C - y * s, z * y * C + x * s, c + z * z * C],
        ],
        dtype=float,
    )

def apply_hinge(coords, seg, rng, max_angle_deg=25):
    s, e = seg
    L = e - s
    if L < 30:
        return coords
    pivot = s + int(rng.integers(10, L - 10))
    axis = rng.normal(size=3)
    ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
    R = _rotmat(axis, ang)
    X = coords.copy()
    p0 = X[pivot].copy()
    X[pivot + 1 : e] = (X[pivot + 1 : e] - p0) @ R.T + p0
    return X

def jitter_chains(coords, segments, rng, max_angle_deg=12, max_trans=1.5):
    X = coords.copy()
    global_center = X.mean(axis=0, keepdims=True)
    for (s, e) in segments:
        axis = rng.normal(size=3)
        ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
        R = _rotmat(axis, ang)
        shift = rng.normal(size=3)
        shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0.0, max_trans))
        c = X[s:e].mean(axis=0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    X -= X.mean(axis=0, keepdims=True) - global_center
    return X

def smooth_wiggle(coords, segments, rng, amp=0.8):
    X = coords.copy()
    for (s, e) in segments:
        L = e - s
        if L < 20:
            continue
        n_ctrl = 6
        ctrl_x = np.linspace(0, L - 1, n_ctrl)
        ctrl_disp = rng.normal(0, amp, size=(n_ctrl, 3))
        t = np.arange(L)
        disp = np.vstack([np.interp(t, ctrl_x, ctrl_disp[:, k]) for k in range(3)]).T
        X[s:e] += disp
    return X

# -----------------------------
# 9) Main prediction function (with ensemble centroid)
# -----------------------------
def predict_rna_structures(row, train_seqs_df, train_coords_dict, n_predictions=5):
    tid = row["target_id"]
    seq = row["sequence"]

    segments = test_segs_map.get(tid, [(0, len(seq))])

    # Candidate pool top_n=30
    cands = find_similar_sequences(
        query_seq=seq, train_seqs_df=train_seqs_df, train_coords_dict=train_coords_dict, top_n=30
    )

    predictions = []
    used = set()

    for i in range(n_predictions):
        seed = (abs(hash(tid)) + i * 10007) % (2**32)
        rng = np.random.default_rng(seed)

        if not cands:
            # Hard fallback: straight line per chain segment
            coords = np.zeros((len(seq), 3), dtype=float)
            for (s, e) in segments:
                for j in range(s + 1, e):
                    coords[j] = coords[j - 1] + [5.95, 0, 0]
            predictions.append(coords)
            continue

        # Template choice
        if i == 0:
            t_id, t_seq, sim, t_coords = cands[0]
        else:
            K = min(12, len(cands))
            sims = np.array([cands[k][2] for k in range(K)], float)
            w = np.exp((sims - sims.max()) / 0.08)
            for k in range(K):
                if cands[k][0] in used:
                    w[k] *= 0.10
            w = w / (w.sum() + 1e-12)
            k = int(rng.choice(np.arange(K), p=w))
            t_id, t_seq, sim, t_coords = cands[k]

        used.add(t_id)

        # Transfer coords
        adapted = adapt_template_to_query(query_seq=seq, template_seq=t_seq, template_coords=t_coords)

        # Diversity transforms
        if i == 0:
            X = adapted
        elif i == 1:
            X = adapted + rng.normal(0, max(0.01, (0.40 - sim) * 0.06), adapted.shape)
        elif i == 2:
            longest = max(segments, key=lambda se: se[1] - se[0])
            X = apply_hinge(adapted, longest, rng, max_angle_deg=22)
        elif i == 3:
            X = jitter_chains(adapted, segments, rng, max_angle_deg=10, max_trans=1.0)
        else:
            X = smooth_wiggle(adapted, segments, rng, amp=0.7)

        refined = adaptive_rna_constraints(X, tid, confidence=sim, passes=2)
        predictions.append(refined)

    # ★★ 新增: Ensemble centroid替换最弱预测
    if len(predictions) >= 3:
        centroid = ensemble_centroid(predictions)
        if centroid is not None and np.isfinite(centroid).all():
            refined_centroid = adaptive_rna_constraints(centroid, tid, confidence=0.9, passes=2)
            
            # 计算各预测的置信度分数
            scores = [hybrid_confidence(p) for p in predictions]
            worst_idx = np.argmin(scores)
            centroid_score = hybrid_confidence(refined_centroid)
            
            # 如果centroid更好，替换最差的预测
            if centroid_score > scores[worst_idx]:
                predictions[worst_idx] = refined_centroid

    return predictions

# -----------------------------
# 10) Submission writer
# -----------------------------
all_predictions = []
start_time = time.time()

for idx, row in test_seqs.iterrows():
    if idx % 10 == 0:
        print(f"Processing {idx} | {time.time() - start_time:.1f}s")
    tid = row["target_id"]
    seq = row["sequence"]

    # ★★ 使用扩展后的模板池
    preds = predict_rna_structures(row, all_train_seqs, all_train_coords, n_predictions=5)

    L = len(seq)
    for p in preds:
        assert isinstance(p, np.ndarray) and p.shape == (L, 3), f"Bad pred shape for {tid}"
        assert np.isfinite(p).all(), f"Non-finite coords in {tid}"

    for j in range(L):
        res = {"ID": f"{tid}_{j+1}", "resname": seq[j], "resid": j + 1}
        for i in range(5):
            res[f"x_{i+1}"], res[f"y_{i+1}"], res[f"z_{i+1}"] = preds[i][j]
        all_predictions.append(res)

sub = pd.DataFrame(all_predictions)
cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, 6) for c in ["x", "y", "z"]]
coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]
sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
sub[cols].to_csv("submission.csv", index=False)

print(f"\n✓ submission.csv saved!")
print(f"Total time: {time.time() - start_time:.1f}s")
print(f"Template pool used: {len(all_train_coords)} structures")

Processing /kaggle/input/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl'

✓ Template pool (train+val): 5744 structures

★★ External template roots NOT found - using train+val only

★★ Final template pool: 5744 structures
Processing 0 | 0.0s
Processing 10 | 136.0s
Processing 20 | 139.2s

✓ submission.csv saved!
Total time: 147.7s
Template pool used: 5744 structures
